# Bronze Data Quality & Consistency Check
Notebook này dùng để kiểm tra tính nhất quán giữa dữ liệu log raw (`JSONL`) sinh ra từ Simulator và dữ liệu đã ingest vào `Delta Bronze` trên MinIO.

**Các bước check độ tối ưu (Reconciliation) cho Data Pipeline:**
1. **Row Count Check:** Đếm số dòng raw (sau khi áp dụng rule lọc) và so sánh với tổng số dòng trong Delta.
2. **Uniqueness Check:** Đảm bảo không có ID bị trùng lặp trong Delta (tránh trường hợp duplicate khi append/merge sai cách).
3. **Data Loss Check (Anti-Join):** Dùng `subtract()` để tìm tập các record có trong raw hợp lệ nhưng lại rơi rớt mất khi ghi vào Delta.
4. **Metric Consistency:** Aggregate một chỉ số quan trọng (vd: tổng `fare_vnd`) nhằm đảm bảo không có sự toàn vẹn nào bị phá vỡ giữa raw và bronze.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, sum as _sum

# Cấu hình Access MinIO
MINIO_ENDPOINT = "http://minio:9000"
MINIO_KEY = "minioadmin"
MINIO_SECRET = "minioadmin123"

# Khởi tạo Spark gắn với Delta & MinIO để đọc cả Raw (JSONL) và Bronze (Delta)
spark = (
    SparkSession.builder
    .appName("rideflow_data_check")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("✅ Spark Session cho Data Check đã sẵn sàng!")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/23 17:26:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ Spark Session cho Data Check đã sẵn sàng!


In [2]:
import os

# === TEST CỤ THỂ CHO BẢNG TRIPS ===
# Bạn thay đổi TARGET_DATE tương ứng với file phân vùng mà bạn muốn verify.
TARGET_DATE = "2026-03-24"
RAW_TRIPS_PATH = os.path.abspath(f"../spark-data/raw/trips/date={TARGET_DATE}/data.jsonl")
BRONZE_TRIPS_PATH = "s3a://rideflow/bronze/trips"

print(f"📌 Đang kiểm tra bảng TRIPS cho ngày: {TARGET_DATE}")

# 1. Đọc Raw Data trực tiếp từ thư mục Raw
try:
    raw_df = spark.read.json(RAW_TRIPS_PATH)
except Exception as e:
    print(f"Không tìm thấy raw data. Vui lòng check: {RAW_TRIPS_PATH}")
    raw_df = spark.createDataFrame([], schema="trip_id string, rider_id string, status string, city string, fare_vnd int")

# Giả lập logic Rule Lọc Mảng (Clean) y như bên trong ingest_trips.py
raw_valid_df = (
    raw_df.dropDuplicates(["trip_id"])
          .filter(col("status").isin(["completed", "cancelled", "no_driver"]))
          .filter(col("city").isin(["HCMC", "HANOI"]))
          .filter(col("trip_id").isNotNull() & col("rider_id").isNotNull())
)

raw_count = raw_df.count()
raw_valid_count = raw_valid_df.count()

# 2. Đọc Bronze Data từ Delta Lake
try:
    bronze_df = spark.read.format("delta").load(BRONZE_TRIPS_PATH).filter(col("ingest_date") == TARGET_DATE)
    bronze_count = bronze_df.count()
except Exception as e:
    print(f"⚠️ Không tìm thấy bảng Delta tại {BRONZE_TRIPS_PATH} hoặc chưa có data phần vùng ngày {TARGET_DATE}.")
    bronze_count = 0
    bronze_df = None

📌 Đang kiểm tra bảng TRIPS cho ngày: 2026-03-24


In [3]:
print("=== 1. ROW COUNT CHECK ===")
print(f"  📊 Raw Ban Đầu:    {raw_count}")
print(f"  ✅ Raw Hợp Lệ:     {raw_valid_count}")
print(f"  🧊 Delta Bronze:   {bronze_count}")

if raw_valid_count == bronze_count and bronze_count > 0:
    print("  => 🟢 PASSED: Số lượng dòng Raw hợp lệ khớp tuyệt đối với Delta!")
else:
    print("  => 🔴 FAILED: Số lượng không khớp!")


print("\n=== 2. UNIQUENESS CHECK ===")
if bronze_df is not None and bronze_count > 0:
    bronze_unique_count = bronze_df.select("trip_id").distinct().count()
    if bronze_unique_count == bronze_count:
        print("  => 🟢 PASSED: 100% trip_id trong Delta là duy nhất.")
    else:
        print(f"  => 🔴 FAILED: Có {bronze_count - bronze_unique_count} trip_id bị trùng lặp trong Delta (cần check lại Rule Append/Merge).")


print("\n=== 3. DATA LOSS CHECK (ANTI-JOIN) ===")
if bronze_df is not None and bronze_count > 0:
    missing_in_bronze = raw_valid_df.select("trip_id").subtract(bronze_df.select("trip_id")).count()
    if missing_in_bronze == 0:
        print("  => 🟢 PASSED: Không có trip_id hợp lệ nào bị rớt khỏi quá trình map-reduce/write.")
    else:
        print(f"  => 🔴 FAILED: Bị mất {missing_in_bronze} trip(s) khi đẩy vào Delta!")


print("\n=== 4. METRIC CONSISTENCY (FARE_VND) ===")
if bronze_df is not None and bronze_count > 0:
    raw_fare = raw_valid_df.select(_sum("fare_vnd")).collect()[0][0] or 0
    bronze_fare = bronze_df.select(_sum("fare_vnd")).collect()[0][0] or 0
    print(f"  💰 Tổng doanh thu Raw:   {raw_fare:,} VNĐ")
    print(f"  🪙 Tổng doanh thu Delta: {bronze_fare:,} VNĐ")
    
    if raw_fare == bronze_fare:
        print("  => 🟢 PASSED: Tổng doanh thu hoàn toàn khớp giữa Data Lake và file Log Raw.")
    else:
        print("  => 🔴 FAILED: Tổng doanh thu bị LỆCH!")

=== 1. ROW COUNT CHECK ===
  📊 Raw Ban Đầu:    2023
  ✅ Raw Hợp Lệ:     2023
  🧊 Delta Bronze:   2023
  => 🟢 PASSED: Số lượng dòng Raw hợp lệ khớp tuyệt đối với Delta!

=== 2. UNIQUENESS CHECK ===


  => 🟢 PASSED: 100% trip_id trong Delta là duy nhất.

=== 3. DATA LOSS CHECK (ANTI-JOIN) ===


  => 🟢 PASSED: Không có trip_id hợp lệ nào bị rớt khỏi quá trình map-reduce/write.

=== 4. METRIC CONSISTENCY (FARE_VND) ===


  💰 Tổng doanh thu Raw:   70,164,000 VNĐ
  🪙 Tổng doanh thu Delta: 70,164,000 VNĐ
  => 🟢 PASSED: Tổng doanh thu hoàn toàn khớp giữa Data Lake và file Log Raw.
